# 04D1E_v5 — LANDFIRE Schema Repair and Finalize

**Build ID: `04D1E_V5_LANDFIRE_SCHEMA_REPAIR_FINALIZE`**

This notebook does **not** rerun LANDFIRE extraction.

It repairs only the narrow schema edge case discovered after the 04D1E V3
recovery:

- 962 / 974 block shards are valid;
- 12 shards exist but report `missing_features:472`;
- no production block failed;
- all newly computed context requests were covered by their raster tiles.

## Safety rule

A shard is repaired **only** when:

1. every missing frozen-V5 feature has `direction_required=True`; and
2. every unit in the shard has
   `landfire_directional_context_valid == False`; and
3. there are no unexpected `landfire_*` columns; and
4. row count and exact expected `unit_uid` membership are correct.

For such a block, the missing directional features are structurally undefined
by design. They are materialized as explicit `NaN` columns so every production
shard has the exact same frozen V5 schema.

If any condition fails, the notebook refuses to patch that shard.

## 0. Configuration

## V5 correction — Windows-safe atomic shard replacement

V4 correctly validated each temporary repaired Parquet shard, but a
`pyarrow.parquet.ParquetFile` object can keep the temporary file open on
Windows. Renaming that still-open `.repairing` file can then raise `WinError 32`.

V5:

1. adds all missing all-NaN directional fields in one `pd.concat`;
2. removes any stale `.repairing` file;
3. writes the temporary repaired shard;
4. validates it with `pq.read_metadata(...)`;
5. atomically replaces the original only after validation.

No LANDFIRE extraction or scientific feature definition changes.


In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = "04D1E_V5_LANDFIRE_SCHEMA_REPAIR_FINALIZE"

from pathlib import Path
import json
import math
import os
import shutil
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

PROJECT_ROOT = Path(
    os.environ.get("FIRELINE_PROJECT_ROOT", ".")
)

TERRAIN_UNIT_INDEX_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "terrain_3dep_production_v1" /
    "terrain_operational_unit_index_v1.parquet"
)

FEATURE_DICTIONARY_PATH = (
    PROJECT_ROOT / "outputs" /
    "landfire_feature_architecture_v5" /
    "LANDFIRE_FEATURE_DICTIONARY_V5.csv"
)

OUTPUT_ROOT = (
    PROJECT_ROOT / "outputs" /
    "landfire_resolved_vintage_full_production_v1"
)

SHARD_ROOT = (
    PROJECT_ROOT / "data" / "processed" /
    "landfire_resolved_vintage_full_production_v1" /
    "feature_shards"
)

APPROACH_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "fireline_extended_exact_approach_v3" /
    "extended_exact_approach_features_v3.parquet"
)

for p in [
    TERRAIN_UNIT_INDEX_PATH,
    FEATURE_DICTIONARY_PATH,
    SHARD_ROOT,
    APPROACH_PATH,
]:
    assert p.exists(), p

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

VERSION_BY_FIRE_YEAR = {
    2020: "LF2016R",
    2021: "LF2019L",
    2022: "LF2020",
    2023: "LF2022",
    2024: "LF2023",
}

RESOLVED_VERSIONS = [
    "LF2016R",
    "LF2022",
    "LF2023",
]

print("=" * 100)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("=" * 100)

## 1. Load frozen universe and V5 schema

In [ ]:
unit_index = pd.read_parquet(
    TERRAIN_UNIT_INDEX_PATH
)

approach = pd.read_parquet(
    APPROACH_PATH,
    columns=[
        "unit_uid",
        "approach_azimuth_deg",
        "approach_direction_reliable",
    ],
)

feature_dictionary = pd.read_csv(
    FEATURE_DICTIONARY_PATH
)

assert unit_index["unit_uid"].is_unique
assert approach["unit_uid"].is_unique
assert feature_dictionary["feature"].is_unique

frozen_features = set(
    feature_dictionary["feature"].astype(str)
)

direction_required_map = (
    feature_dictionary.set_index(
        "feature"
    )["direction_required"]
    .fillna(False)
    .astype(bool)
    .to_dict()
)

base = unit_index.copy()

base["Year"] = pd.to_numeric(
    base["Year"],
    errors="raise",
).astype(int)

base["landfire_version_key"] = (
    base["Year"].map(
        VERSION_BY_FIRE_YEAR
    )
)

overlap = [
    c
    for c in [
        "approach_azimuth_deg",
        "approach_direction_reliable",
    ]
    if c in base.columns
]

base = base.drop(
    columns=overlap,
    errors="ignore",
)

base = base.merge(
    approach,
    on="unit_uid",
    how="left",
    validate="one_to_one",
)

base[
    "landfire_directional_context_valid"
] = (
    base[
        "approach_direction_reliable"
    ].fillna(False)
    & pd.to_numeric(
        base["approach_azimuth_deg"],
        errors="coerce",
    ).notna()
)

resolved_base = base[
    base["landfire_version_key"].isin(
        RESOLVED_VERSIONS
    )
].copy()

block_manifest = (
    resolved_base.groupby(
        [
            "landfire_version_key",
            "terrain_block_id",
        ],
        as_index=False,
    )
    .agg(
        expected_units=(
            "unit_uid",
            "size",
        ),
    )
)

print(
    "Resolved production units:",
    f"{len(resolved_base):,}",
)

print(
    "Resolved production blocks:",
    f"{len(block_manifest):,}",
)

print(
    "Frozen V5 features:",
    f"{len(frozen_features):,}",
)

print(
    "Direction-required frozen features:",
    f"{sum(direction_required_map.values()):,}",
)

## 2. Locate and classify every production shard

In [ ]:
def safe_token(value, n=16):
    import hashlib
    return hashlib.sha1(
        str(value).encode("utf-8")
    ).hexdigest()[:n]


def expected_shard_path(
    version_key,
    block_id,
):
    return (
        SHARD_ROOT /
        f"{version_key}__{safe_token(block_id)}.parquet"
    )


def classify_shard(
    version_key,
    block_id,
    expected_units,
):
    path = expected_shard_path(
        version_key,
        block_id,
    )

    if not path.exists():
        return {
            "version_key": version_key,
            "block_id": block_id,
            "expected_units": expected_units,
            "status": "missing_shard",
            "missing_count": np.nan,
            "unexpected_count": np.nan,
            "shard_path": str(path),
        }

    pf = pq.ParquetFile(
        path
    )

    cols = set(
        pf.schema.names
    )

    missing = sorted(
        frozen_features - cols
    )

    unexpected = sorted(
        c
        for c in cols
        if c.startswith("landfire_")
        and c not in frozen_features
    )

    return {
        "version_key": version_key,
        "block_id": block_id,
        "expected_units": int(expected_units),
        "observed_rows": int(
            pf.metadata.num_rows
        ),
        "status": (
            "valid"
            if (
                not missing
                and not unexpected
                and int(pf.metadata.num_rows)
                    == int(expected_units)
            )
            else "invalid"
        ),
        "missing_count": len(missing),
        "unexpected_count": len(unexpected),
        "missing_features": missing,
        "unexpected_features": unexpected,
        "shard_path": str(path),
    }


audit_rows = []

for _, row in block_manifest.iterrows():
    audit_rows.append(
        classify_shard(
            row["landfire_version_key"],
            row["terrain_block_id"],
            int(row["expected_units"]),
        )
    )

pre_repair_audit = pd.DataFrame(
    audit_rows
)

display(
    pre_repair_audit.groupby(
        [
            "version_key",
            "status",
            "missing_count",
        ],
        dropna=False,
        as_index=False,
    ).agg(
        blocks=("block_id", "size"),
        units=("expected_units", "sum"),
    )
)

pre_repair_audit.to_pickle(
    OUTPUT_ROOT /
    "SCHEMA_REPAIR_PRE_AUDIT_V5.pkl"
)

## 3. Strict eligibility audit for invalid shards

In [ ]:
invalid = pre_repair_audit[
    pre_repair_audit["status"]
    != "valid"
].copy()

eligibility_rows = []

for _, r in invalid.iterrows():
    version_key = r[
        "version_key"
    ]

    block_id = r[
        "block_id"
    ]

    path = Path(
        r["shard_path"]
    )

    row = {
        "version_key": version_key,
        "block_id": block_id,
        "expected_units": int(
            r["expected_units"]
        ),
        "status_before": r[
            "status"
        ],
        "missing_count": r.get(
            "missing_count",
            np.nan,
        ),
        "eligible_for_nan_schema_repair": False,
        "reason": None,
        "directional_valid_units": np.nan,
    }

    if not path.exists():
        row[
            "reason"
        ] = "shard_missing"
        eligibility_rows.append(
            row
        )
        continue

    missing = list(
        r["missing_features"]
    )

    unexpected = list(
        r["unexpected_features"]
    )

    if unexpected:
        row[
            "reason"
        ] = (
            f"unexpected_landfire_columns:{len(unexpected)}"
        )
        eligibility_rows.append(
            row
        )
        continue

    if not missing:
        row[
            "reason"
        ] = "no_missing_features"
        eligibility_rows.append(
            row
        )
        continue

    nondirectional_missing = [
        f
        for f in missing
        if not direction_required_map.get(
            f,
            False,
        )
    ]

    if nondirectional_missing:
        row[
            "reason"
        ] = (
            "missing_non_directional_features:"
            f"{len(nondirectional_missing)}"
        )
        eligibility_rows.append(
            row
        )
        continue

    # Read only identity + directional QA from shard.
    needed = [
        "unit_uid",
        "landfire_version_key",
        "terrain_block_id",
        "landfire_directional_context_valid",
    ]

    x = pd.read_parquet(
        path,
        columns=needed,
    )

    if len(x) != int(
        r["expected_units"]
    ):
        row[
            "reason"
        ] = "row_count_mismatch"
        eligibility_rows.append(
            row
        )
        continue

    if not x[
        "unit_uid"
    ].is_unique:
        row[
            "reason"
        ] = "unit_uid_not_unique"
        eligibility_rows.append(
            row
        )
        continue

    expected_uids = set(
        resolved_base.loc[
            (
                resolved_base[
                    "landfire_version_key"
                ]
                == version_key
            )
            & (
                resolved_base[
                    "terrain_block_id"
                ]
                == block_id
            ),
            "unit_uid",
        ].astype(str)
    )

    observed_uids = set(
        x[
            "unit_uid"
        ].astype(str)
    )

    if observed_uids != expected_uids:
        row[
            "reason"
        ] = "unit_uid_set_mismatch"
        eligibility_rows.append(
            row
        )
        continue

    if not x[
        "landfire_version_key"
    ].astype(str).eq(
        str(version_key)
    ).all():
        row[
            "reason"
        ] = "version_mismatch"
        eligibility_rows.append(
            row
        )
        continue

    if not x[
        "terrain_block_id"
    ].astype(str).eq(
        str(block_id)
    ).all():
        row[
            "reason"
        ] = "block_mismatch"
        eligibility_rows.append(
            row
        )
        continue

    directional_valid_units = int(
        x[
            "landfire_directional_context_valid"
        ]
        .fillna(False)
        .astype(bool)
        .sum()
    )

    row[
        "directional_valid_units"
    ] = directional_valid_units

    if directional_valid_units != 0:
        row[
            "reason"
        ] = (
            "block_contains_direction_valid_units"
        )
        eligibility_rows.append(
            row
        )
        continue

    row[
        "eligible_for_nan_schema_repair"
    ] = True

    row[
        "reason"
    ] = (
        "all_missing_features_direction_required_"
        "and_zero_direction_valid_units"
    )

    eligibility_rows.append(
        row
    )


eligibility = pd.DataFrame(
    eligibility_rows
)

display(
    eligibility
)

print()
print(
    "Eligible repairs:",
    int(
        eligibility[
            "eligible_for_nan_schema_repair"
        ].sum()
    ),
    "/",
    len(eligibility),
)

eligibility.to_csv(
    OUTPUT_ROOT /
    "SCHEMA_REPAIR_ELIGIBILITY_V5.csv",
    index=False,
)

## 4. Verify the recurring 472-feature pattern

In [ ]:
eligible_rows = eligibility[
    eligibility[
        "eligible_for_nan_schema_repair"
    ]
].copy()

if len(eligible_rows):
    missing_sets = []

    for _, e in eligible_rows.iterrows():
        a = pre_repair_audit[
            (
                pre_repair_audit[
                    "version_key"
                ]
                == e["version_key"]
            )
            & (
                pre_repair_audit[
                    "block_id"
                ]
                == e["block_id"]
            )
        ].iloc[0]

        missing_sets.append(
            frozenset(
                a[
                    "missing_features"
                ]
            )
        )

    unique_missing_sets = set(
        missing_sets
    )

    print(
        "Unique missing-feature sets:",
        len(unique_missing_sets),
    )

    print(
        "Missing feature counts:",
        sorted(
            {
                len(s)
                for s in unique_missing_sets
            }
        ),
    )

    assert len(
        unique_missing_sets
    ) == 1

    common_missing = set(
        next(
            iter(
                unique_missing_sets
            )
        )
    )

    assert all(
        direction_required_map[f]
        for f in common_missing
    )

    print(
        "COMMON DIRECTION-ONLY MISSING SCHEMA:",
        len(common_missing),
        "features",
    )

else:
    common_missing = set()

print(
    "DIRECTION-ONLY SCHEMA PATTERN QA: PASS"
)

## 5. Repair eligible shards atomically

In [ ]:

repair_log_rows = []

for _, e in eligibility.iterrows():

    if not bool(
        e["eligible_for_nan_schema_repair"]
    ):
        continue

    version_key = e["version_key"]
    block_id = e["block_id"]

    path = expected_shard_path(
        version_key,
        block_id,
    )

    a = pre_repair_audit[
        (
            pre_repair_audit["version_key"]
            == version_key
        )
        & (
            pre_repair_audit["block_id"]
            == block_id
        )
    ].iloc[0]

    missing = list(
        a["missing_features"]
    )

    # Tiny directionless edge shard.
    x = pd.read_parquet(
        path
    )

    already_present = [
        feature
        for feature in missing
        if feature in x.columns
    ]

    if already_present:
        raise RuntimeError(
            f"{version_key}/{block_id}: "
            f"{len(already_present)} proposed repair fields "
            "are unexpectedly already present."
        )

    nondirectional = [
        feature
        for feature in missing
        if not direction_required_map.get(
            feature,
            False,
        )
    ]

    if nondirectional:
        raise RuntimeError(
            f"{version_key}/{block_id}: "
            f"refusing to add {len(nondirectional)} "
            "non-directional feature(s)."
        )

    # Add all structurally unavailable directional features in one operation.
    # This avoids DataFrame fragmentation from ~472 repeated inserts.
    nan_block = pd.DataFrame(
        np.nan,
        index=x.index,
        columns=missing,
        dtype=float,
    )

    x = pd.concat(
        [x, nan_block],
        axis=1,
    ).copy()

    # Identity / schema validation before write.
    assert len(x) == int(
        e["expected_units"]
    )

    assert x["unit_uid"].is_unique

    after_cols = set(
        x.columns
    )

    missing_after = sorted(
        frozen_features
        - after_cols
    )

    unexpected_after = sorted(
        c
        for c in after_cols
        if c.startswith("landfire_")
        and c not in frozen_features
    )

    if missing_after:
        raise RuntimeError(
            f"{version_key}/{block_id}: "
            f"still missing {len(missing_after)} frozen features"
        )

    if unexpected_after:
        raise RuntimeError(
            f"{version_key}/{block_id}: "
            f"{len(unexpected_after)} unexpected LANDFIRE features"
        )

    tmp = path.with_suffix(
        path.suffix + ".repairing"
    )

    # A prior failed V4 run may have left the temporary file behind.
    if tmp.exists():
        try:
            tmp.unlink()
        except PermissionError as exc:
            raise RuntimeError(
                f"{version_key}/{block_id}: stale temporary file is "
                f"still locked: {tmp}. Restart the Jupyter kernel and "
                "close any data viewer/Explorer preview holding it, "
                "then rerun."
            ) from exc

    x.to_parquet(
        tmp,
        index=False,
    )

    # Windows-safe validation: read metadata and let the call return before
    # attempting replacement. Do NOT retain a ParquetFile object here.
    tmp_meta = pq.read_metadata(
        tmp
    )

    if int(
        tmp_meta.num_rows
    ) != len(x):
        raise RuntimeError(
            "Temporary repaired shard row count mismatch."
        )

    tmp_cols = set(
        tmp_meta.schema.names
    )

    missing_tmp = sorted(
        frozen_features
        - tmp_cols
    )

    unexpected_tmp = sorted(
        c
        for c in tmp_cols
        if c.startswith("landfire_")
        and c not in frozen_features
    )

    if missing_tmp:
        raise RuntimeError(
            f"{version_key}/{block_id}: temporary shard "
            f"still missing {len(missing_tmp)} frozen features."
        )

    if unexpected_tmp:
        raise RuntimeError(
            f"{version_key}/{block_id}: temporary shard has "
            f"{len(unexpected_tmp)} unexpected LANDFIRE fields."
        )

    # Explicitly release metadata and temporary helper objects before the
    # Windows atomic replacement.
    del tmp_meta
    del nan_block

    try:
        os.replace(
            str(tmp),
            str(path),
        )
    except PermissionError as exc:
        raise RuntimeError(
            f"{version_key}/{block_id}: Windows still has a file handle "
            f"open during replacement. Source={tmp}, Target={path}. "
            "Restart the kernel and close any VS Code/Explorer Parquet "
            "preview, then rerun."
        ) from exc

    repair_log_rows.append({
        "version_key": version_key,
        "block_id": block_id,
        "units": len(x),
        "features_materialized_as_nan": len(missing),
        "shard_path": str(path),
        "repaired_utc": datetime.now(
            timezone.utc
        ).isoformat(),
    })

    print(
        "REPAIRED:",
        version_key,
        block_id,
        "| units:",
        len(x),
        "| NaN schema columns added:",
        len(missing),
    )


repair_log = pd.DataFrame(
    repair_log_rows
)

display(
    repair_log
)

repair_log.to_csv(
    OUTPUT_ROOT /
    "SCHEMA_REPAIR_LOG_V5.csv",
    index=False,
)


## 6. Full post-repair shard audit

In [ ]:
post_rows = []

for _, row in block_manifest.iterrows():
    post_rows.append(
        classify_shard(
            row[
                "landfire_version_key"
            ],
            row[
                "terrain_block_id"
            ],
            int(
                row[
                    "expected_units"
                ]
            ),
        )
    )

post_audit = pd.DataFrame(
    post_rows
)

display(
    post_audit.groupby(
        [
            "version_key",
            "status",
        ],
        as_index=False,
    ).agg(
        blocks=("block_id", "size"),
        units=("expected_units", "sum"),
    )
)

invalid_after = post_audit[
    post_audit[
        "status"
    ] != "valid"
].copy()

display(
    invalid_after
)

post_audit.to_pickle(
    OUTPUT_ROOT /
    "SCHEMA_REPAIR_POST_AUDIT_V5.pkl"
)

invalid_after.to_csv(
    OUTPUT_ROOT /
    "SCHEMA_REPAIR_INVALID_AFTER_V5.csv",
    index=False,
)

## 7. Final resolved-vintage production checks

In [ ]:
expected_blocks = len(
    block_manifest
)

expected_units = int(
    block_manifest[
        "expected_units"
    ].sum()
)

valid_after = post_audit[
    post_audit[
        "status"
    ] == "valid"
]

valid_blocks = len(
    valid_after
)

valid_units = int(
    valid_after[
        "expected_units"
    ].sum()
)

checks = pd.DataFrame([
    {
        "check": (
            "all_expected_blocks_have_valid_shards"
        ),
        "pass": (
            valid_blocks
            == expected_blocks
        ),
        "observed": valid_blocks,
        "expected": expected_blocks,
    },
    {
        "check": (
            "all_expected_units_covered"
        ),
        "pass": (
            valid_units
            == expected_units
        ),
        "observed": valid_units,
        "expected": expected_units,
    },
    {
        "check": (
            "no_invalid_shards_after_repair"
        ),
        "pass": (
            len(
                invalid_after
            ) == 0
        ),
        "observed": len(
            invalid_after
        ),
        "expected": 0,
    },
    {
        "check": (
            "exact_frozen_v5_schema_all_shards"
        ),
        "pass": (
            valid_blocks
            == expected_blocks
        ),
        "observed": valid_blocks,
        "expected": expected_blocks,
    },
])

display(
    checks
)

FINAL_PASS = bool(
    checks[
        "pass"
    ].all()
)

print()
print(
    "04D1E RESOLVED-VINTAGE PRODUCTION:",
    (
        "PASS / COMPLETE"
        if FINAL_PASS
        else "INCOMPLETE / REVIEW REQUIRED"
    ),
)

checks.to_csv(
    OUTPUT_ROOT /
    "PRODUCTION_FINAL_CHECKS_V5.csv",
    index=False,
)

## 8. Write final repair report

In [ ]:
def tomd(df):
    try:
        return df.to_markdown(
            index=False
        )
    except Exception:
        return df.to_string(
            index=False
        )


report = f"""# 04D1E LANDFIRE Resolved-Vintage Production — Schema Repair Finalization

Build: `{NOTEBOOK_BUILD_ID}`

Generated UTC: {datetime.now(timezone.utc).isoformat()}

## Frozen production universe

Blocks: {expected_blocks:,}

Units: {expected_units:,}

Versions: {", ".join(RESOLVED_VERSIONS)}

## Repair eligibility

{tomd(eligibility)}

## Repair log

{tomd(repair_log) if len(repair_log) else "No repairs were required."}

## Invalid shards after repair

{tomd(invalid_after) if len(invalid_after) else "None."}

## Final checks

{tomd(checks)}

## Decision

`{"PASS / COMPLETE" if FINAL_PASS else "INCOMPLETE / REVIEW REQUIRED"}`

The repair materializes only structurally unavailable directional fields as
explicit NaN columns for blocks with zero reliable directional context.

No LANDFIRE raster was redownloaded.
No extracted value was recomputed.
No valid existing feature value was modified.
The frozen 04D1C V5 feature definition is unchanged.
"""

REPORT_PATH = (
    OUTPUT_ROOT /
    "LANDFIRE_RESOLVED_VINTAGE_SCHEMA_REPAIR_REPORT_V5.md"
)

REPORT_PATH.write_text(
    report,
    encoding="utf-8",
)

print(report)
print()
print(
    "Saved:",
    REPORT_PATH,
)

## 9. Output inventory

In [ ]:
print("=" * 100)
print("04D1E V5 REPAIR OUTPUTS")
print("=" * 100)

for p in sorted(
    OUTPUT_ROOT.glob(
        "*V5*"
    )
):
    if p.is_file():
        print(
            f"{p.name:<90}"
            f"{p.stat().st_size/(1024**2):>12.3f} MB"
        )

print()
print(
    "DONE:",
    NOTEBOOK_BUILD_ID,
)